# Sesión 2: Recuperación

En esta sesión construimos perfiles de usuario a partir de los juegos consumidos y los usamos como consultas del sistema de recuperación de información. Cada perfil textual se transforma en un embedding semántico y, como los embeddings de los juegos están normalizados y almacenados en un índice `FAISS IndexFlatIP`, la recuperación equivale a buscar los juegos más similares mediante similitud coseno.

## 2. Imports

In [1]:
import ast
import json
import os
import re

import faiss
import numpy as np
import pandas as pd
from bs4 import BeautifulSoup
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

c:\Users\Willy\Desktop\Master\Sistemas de Recuperación de Información y de Recomendación\Practica\Rag\rag-steam-rag\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. Definición de rutas

In [2]:
USER_ITEMS_PATH = "../data/raw/steam_user.json"
GAMES_DATASET_PATH = "../outputs/games_dataset.csv"
EMBEDDINGS_PATH = "../outputs/embeddings.npy"
FAISS_INDEX_PATH = "../outputs/faiss.index"
OUTPUTS_DIR = "../outputs"

for path in [USER_ITEMS_PATH, GAMES_DATASET_PATH, EMBEDDINGS_PATH, FAISS_INDEX_PATH]:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"No se encontró el archivo {path}. Ejecuta antes la sesión 1 desde la carpeta notebooks/."
        )

os.makedirs(OUTPUTS_DIR, exist_ok=True)

## 4. Cargar resultados de la sesión 1

In [3]:
games_df = pd.read_csv(GAMES_DATASET_PATH)
games_df["item_id"] = games_df["item_id"].astype(str)

item_id_to_name = dict(zip(games_df["item_id"], games_df["item_name"]))
item_id_to_document = dict(zip(games_df["item_id"], games_df["document"]))

embeddings = np.load(EMBEDDINGS_PATH)
index = faiss.read_index(FAISS_INDEX_PATH)

In [4]:
games_df.head()

,item_id,item_name,num_reviews,document
0,251570,7 Days to Die,20,7 days to die this game is awesome its like mi...
1,250420,8BitMMO,20,8bitmmo this game is so much fun i love buildi...
2,113400,APB Reloaded,20,apb reloaded i do hate the fact that i lagged ...
3,346110,ARK: Survival Evolved,20,ark survival evolved so i play ark on and off ...
4,224540,Ace of Spades,20,ace of spades meh awesome great game yeah a pr...


In [5]:
embeddings.shape

(2058, 384)

In [6]:
index.ntotal

2058

## 5. Función `read_json_lines`

In [7]:
def read_json_lines(path, max_lines=None):
    """Lee un archivo pseudo-JSONL línea a línea usando ast.literal_eval."""
    records = []

    with open(path, "r", encoding="utf-8") as f:
        iterator = tqdm(f, desc=f"Leyendo {os.path.basename(path)}", unit="line")

        for i, line in enumerate(iterator):
            if max_lines is not None and i >= max_lines:
                break

            line = line.strip()
            if not line:
                continue

            try:
                records.append(ast.literal_eval(line))
            except (SyntaxError, ValueError):
                continue

    return records

## 6. Función `clean_text`

In [8]:
def clean_text(text):
    """Limpia texto eliminando HTML, saltos de línea y caracteres raros."""
    if text is None:
        return ""

    text = BeautifulSoup(str(text), "html.parser").get_text(" ")
    text = text.lower()
    text = re.sub(r"[\r\n\t]+", " ", text)
    text = re.sub(r"[^\w\s]+", " ", text, flags=re.UNICODE)
    text = re.sub(r"_+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def safe_text_for_display(text):
    """Convierte texto a una versión imprimible en consolas Windows."""
    return str(text).encode("cp1252", errors="replace").decode("cp1252")

## 7. Función `build_user_profiles`

In [9]:
def build_user_profiles(
    user_items_path,
    valid_item_ids,
    item_id_to_document,
    min_items=5,
    max_items_per_user=15,
    max_profile_docs=5,
    max_users=20,
):
    """Construye perfiles de usuario enriquecidos con nombres y documentos de juegos consumidos."""
    records = read_json_lines(user_items_path)
    rows = []

    for user_record in tqdm(records, desc="Construyendo perfiles de usuario"):
        user_id = str(user_record.get("user_id", "")).strip()
        items = user_record.get("items", [])

        if not user_id or not isinstance(items, list):
            continue

        valid_items = []

        for item_data in items:
            if not isinstance(item_data, dict):
                continue

            item_id = str(item_data.get("item_id", "")).strip()
            item_name = str(item_data.get("item_name", "")).strip()
            playtime_value = item_data.get("playtime_forever", 0)

            try:
                playtime_forever = float(playtime_value)
            except (TypeError, ValueError):
                continue

            if not item_id or not item_name:
                continue

            if item_id not in valid_item_ids or playtime_forever <= 0:
                continue

            valid_items.append(
                {
                    "item_id": item_id,
                    "item_name": item_name,
                    "playtime_forever": playtime_forever,
                }
            )

        if not valid_items:
            continue

        valid_items = sorted(
            valid_items,
            key=lambda item: (-item["playtime_forever"], item["item_name"]),
        )[:max_items_per_user]

        if len(valid_items) < min_items:
            continue

        consumed_item_ids = [item["item_id"] for item in valid_items]
        consumed_item_names = [item["item_name"] for item in valid_items]

        profile_docs = [
            item_id_to_document[item_id][:1000]
            for item_id in consumed_item_ids[:max_profile_docs]
            if item_id in item_id_to_document
        ]

        profile_text = clean_text(
            " ".join(consumed_item_names) + " " + " ".join(profile_docs)
        )

        if not profile_text:
            continue

        rows.append(
            {
                "user_id": user_id,
                "num_items": len(consumed_item_ids),
                "profile_text": profile_text,
                "consumed_item_ids": consumed_item_ids,
                "consumed_item_names": consumed_item_names,
            }
        )

        if len(rows) >= max_users:
            break

    columns = [
        "user_id",
        "num_items",
        "profile_text",
        "consumed_item_ids",
        "consumed_item_names",
    ]
    return pd.DataFrame(rows, columns=columns)

## 8. Crear perfiles

In [10]:
valid_item_ids = set(games_df["item_id"].astype(str))

user_profiles_df = build_user_profiles(
    USER_ITEMS_PATH,
    valid_item_ids,
    item_id_to_document,
    min_items=5,
    max_items_per_user=15,
    max_profile_docs=5,
    max_users=20,
)

if user_profiles_df.empty:
    raise ValueError(
        "No se han generado perfiles de usuario. Revisa los datos o reduce min_items."
    )

Leyendo steam_user.json: 88310line [01:26, 1023.06line/s]
Construyendo perfiles de usuario:   0%|          | 21/88310 [00:00<01:15, 1166.32it/s]


In [11]:
user_profiles_df.head()

,user_id,num_items,profile_text,consumed_item_ids,consumed_item_names
0,76561197970982479,15,counter strike global offensive rising storm r...,"[730, 35450, 8930, 1250, 232090, 24960, 24980,...","[Counter-Strike: Global Offensive, Rising Stor..."
1,js41637,15,the elder scrolls v skyrim terraria saints row...,"[72850, 105600, 55230, 620, 238010, 28050, 242...","[The Elder Scrolls V: Skyrim, Terraria, Saints..."
2,evcentric,15,gnomoria fallout new vegas borderlands fallout...,"[224500, 22380, 8980, 377160, 49520, 28050, 10...","[Gnomoria, Fallout: New Vegas, Borderlands, Fa..."
3,Riot-Punch,15,grand theft auto iv street fighter iv the witc...,"[12210, 21660, 292030, 72850, 12750, 8870, 620...","[Grand Theft Auto IV, Street Fighter IV, The W..."
4,doctr,15,counter strike global offensive borderlands 2 ...,"[730, 49520, 311210, 550, 218620, 22380, 37716...","[Counter-Strike: Global Offensive, Borderlands..."


In [12]:
user_profiles_df.shape

(20, 5)

In [13]:
user_profiles_df["num_items"].describe()

count    20.000000
mean     14.750000
std       1.118034
min      10.000000
25%      15.000000
50%      15.000000
75%      15.000000
max      15.000000
Name: num_items, dtype: float64

Se ha enriquecido el perfil del usuario incorporando documentos textuales de los juegos consumidos. De esta forma, la consulta semántica del usuario queda representada en el mismo tipo de espacio textual que los ítems indexados, ya que ambos contienen nombres y contenido textual derivado de reseñas.

Además, la longitud de cada documento añadido al perfil se limita a los primeros 1000 caracteres para reducir ruido y evitar perfiles excesivamente largos.

In [14]:
example_profile = user_profiles_df.iloc[0]
print("user_id:", example_profile["user_id"])
print("\nconsumed_item_names:")
print(safe_text_for_display(example_profile["consumed_item_names"]))
print("\nprofile_text (primeros 1200 caracteres):")
print(safe_text_for_display(example_profile["profile_text"][:1200]))

user_id: 76561197970982479

consumed_item_names:
['Counter-Strike: Global Offensive', 'Rising Storm/Red Orchestra 2 Multiplayer', "Sid Meier's Civilization V", 'Killing Floor', 'Killing Floor 2', 'Battlefield: Bad Company 2', 'Mass Effect 2', 'Day of Defeat: Source', 'Dragon Age: Origins', 'Plants vs. Zombies: Game of the Year', 'XCOM: Enemy Unknown', 'Just Cause 2', 'Borderlands', 'Insurgency', 'FINAL FANTASY VII']

profile_text (primeros 1200 caracteres):
counter strike global offensive rising storm red orchestra 2 multiplayer sid meier s civilization v killing floor killing floor 2 battlefield bad company 2 mass effect 2 day of defeat source dragon age origins plants vs zombies game of the year xcom enemy unknown just cause 2 borderlands insurgency final fantasy vii counter strike global offensive hold shift to win hold ctrl to lose zika do baile best game in the bloody world good game it was very fun i really do recommend anyone who has played any cs or cod like games to give this 

In [15]:
user_profiles_df["profile_text"].str.len().describe()

count      20.000000
mean     5189.350000
std       226.267954
min      4448.000000
25%      5219.750000
50%      5252.500000
75%      5293.500000
max      5322.000000
Name: profile_text, dtype: float64

## 9. Guardar perfiles

In [16]:
user_profiles_path = os.path.join(OUTPUTS_DIR, "user_profiles.csv")

user_profiles_to_save = user_profiles_df.copy()
user_profiles_to_save["consumed_item_ids"] = user_profiles_to_save["consumed_item_ids"].apply(
    lambda values: json.dumps(values, ensure_ascii=False)
)
user_profiles_to_save["consumed_item_names"] = user_profiles_to_save["consumed_item_names"].apply(
    lambda values: json.dumps(values, ensure_ascii=False)
)

user_profiles_to_save.to_csv(user_profiles_path, index=False, encoding="utf-8")
print(f"Perfiles guardados en: {user_profiles_path}")

Perfiles guardados en: ../outputs\user_profiles.csv


## 10. Generar embeddings de perfiles

In [17]:
model = SentenceTransformer("all-MiniLM-L6-v2")

documents = user_profiles_df["profile_text"].tolist()
user_embeddings = model.encode(
    documents,
    convert_to_numpy=True,
    show_progress_bar=True,
)

user_embeddings = np.asarray(user_embeddings, dtype=np.float32)
faiss.normalize_L2(user_embeddings)
user_embeddings.shape

Batches: 100%|██████████| 1/1 [00:00<00:00,  1.55it/s]


(20, 384)

## 11. Recuperación top-k

In [18]:
final_k = 10
search_k = min(50, index.ntotal)

scores, indices = index.search(user_embeddings, search_k)

retrieval_rows = []

for user_position, user_row in user_profiles_df.reset_index(drop=True).iterrows():
    consumed_item_ids = {str(item_id) for item_id in user_row["consumed_item_ids"]}
    consumed_item_names = user_row["consumed_item_names"]
    user_id = user_row["user_id"]
    profile_text = user_row["profile_text"]

    rank = 1
    for score, idx in zip(scores[user_position], indices[user_position]):
        if idx < 0:
            continue

        game_row = games_df.iloc[int(idx)]
        item_id = str(game_row["item_id"])
        item_name = item_id_to_name.get(item_id, game_row["item_name"])

        if item_id in consumed_item_ids:
            continue

        retrieval_rows.append(
            {
                "user_id": user_id,
                "rank": rank,
                "item_id": item_id,
                "item_name": item_name,
                "score": float(score),
                "profile_text": profile_text,
                "consumed_item_names": consumed_item_names,
            }
        )

        rank += 1
        if rank > final_k:
            break

retrieval_results = pd.DataFrame(
    retrieval_rows,
    columns=[
        "user_id",
        "rank",
        "item_id",
        "item_name",
        "score",
        "profile_text",
        "consumed_item_names",
    ],
)

if retrieval_results.empty:
    raise ValueError("No se han generado recomendaciones. Revisa los perfiles o la recuperación.")

retrieval_results.head()

,user_id,rank,item_id,item_name,score,profile_text,consumed_item_names
0,76561197970982479,1,42700,Call of Duty: Black Ops,0.697285,counter strike global offensive rising storm r...,"[Counter-Strike: Global Offensive, Rising Stor..."
1,76561197970982479,2,10090,Call of Duty: World at War,0.692121,counter strike global offensive rising storm r...,"[Counter-Strike: Global Offensive, Rising Stor..."
2,76561197970982479,3,33930,Arma 2: Operation Arrowhead,0.685877,counter strike global offensive rising storm r...,"[Counter-Strike: Global Offensive, Rising Stor..."
3,76561197970982479,4,10,Counter-Strike,0.672896,counter strike global offensive rising storm r...,"[Counter-Strike: Global Offensive, Rising Stor..."
4,76561197970982479,5,202970,Call of Duty: Black Ops II,0.669079,counter strike global offensive rising storm r...,"[Counter-Strike: Global Offensive, Rising Stor..."


## 12. Guardar resultados

In [19]:
retrieval_results_path = os.path.join(OUTPUTS_DIR, "retrieval_results.csv")

retrieval_results_to_save = retrieval_results.copy()
retrieval_results_to_save["consumed_item_names"] = retrieval_results_to_save["consumed_item_names"].apply(
    lambda values: json.dumps(values, ensure_ascii=False)
)

retrieval_results_to_save.to_csv(retrieval_results_path, index=False, encoding="utf-8")
print(f"Resultados guardados en: {retrieval_results_path}")

Resultados guardados en: ../outputs\retrieval_results.csv


## 13. Mostrar ejemplo de recuperación

In [20]:
example_user = user_profiles_df.iloc[0]
example_user_id = example_user["user_id"]

print("user_id:", example_user_id)
print("\nprofile_text:")
print(safe_text_for_display(example_user["profile_text"]))
print("\nconsumed_item_names:")
print(safe_text_for_display(example_user["consumed_item_names"]))

retrieval_results.loc[
    retrieval_results["user_id"] == example_user_id,
    ["rank", "item_name", "score"],
].reset_index(drop=True)

user_id: 76561197970982479

profile_text:
counter strike global offensive rising storm red orchestra 2 multiplayer sid meier s civilization v killing floor killing floor 2 battlefield bad company 2 mass effect 2 day of defeat source dragon age origins plants vs zombies game of the year xcom enemy unknown just cause 2 borderlands insurgency final fantasy vii counter strike global offensive hold shift to win hold ctrl to lose zika do baile best game in the bloody world good game it was very fun i really do recommend anyone who has played any cs or cod like games to give this game a go it may not be like cod but it is still as much fun maybe even better keepitreal sees player tries shoot at the head fails dies vote kick player rage quitsalloha snackbar ? ?? ?? ? ???? ?? ????? ? great game would recommend best game evar 10 10 ign actually such an addictive game really entertaining and extremely competitive at times i d recommend it to you cause its so good thats really it it s a great tdm 

,rank,item_name,score
0,1,Call of Duty: Black Ops,0.697285
1,2,Call of Duty: World at War,0.692121
2,3,Arma 2: Operation Arrowhead,0.685877
3,4,Counter-Strike,0.672896
4,5,Call of Duty: Black Ops II,0.669079
5,6,Red Crucible: Firestorm,0.661126
6,7,Dead Sky,0.656392
7,8,The Culling,0.656224
8,9,Call of Juarez Gunslinger,0.655868
9,10,Cortex Command,0.654048


## 14. Análisis exploratorio simple

In [21]:
print("Número total de recomendaciones generadas:", len(retrieval_results))
print("Número de usuarios con recomendaciones:", retrieval_results["user_id"].nunique())
print("Score medio global:", retrieval_results["score"].mean())

Número total de recomendaciones generadas: 200
Número de usuarios con recomendaciones: 20
Score medio global: 0.6138978073000908


In [22]:
retrieval_results.groupby("user_id")["score"].mean().sort_values(ascending=False)

user_id
cadmusthreepointoh          0.679310
76561197970982479           0.670091
MinxIsBetterThanPotatoes    0.669763
doctr                       0.663419
MeaTCompany                 0.660023
76561198156664158           0.654959
76561198077246154           0.650149
js41637                     0.648239
76561198089393905           0.644007
WeiEDKrSat                  0.642174
thequeenpanda               0.627482
jorellpogi                  0.624928
NitemarePK                  0.605429
evcentric                   0.603848
Riot-Punch                  0.599090
maplemage                   0.589338
76561198070234207           0.568320
corrupted_soul              0.566545
themanwich                  0.545528
76561198064823719           0.365313
Name: score, dtype: float64

In [23]:
retrieval_results["item_name"].value_counts().head(10)

item_name
Takedown: Red Sabre            9
Call of Duty: Black Ops        8
DETOUR                         8
Call of Duty: World at War     7
Counter-Strike                 7
Battlefield 2                  7
The Culling                    7
Arma 2: Operation Arrowhead    6
Red Crucible: Firestorm        5
Call of Duty: Black Ops II     5
Name: count, dtype: int64

## 15. Comprobaciones de calidad

In [24]:
user_to_consumed_ids = {
    row["user_id"]: {str(item_id) for item_id in row["consumed_item_ids"]}
    for _, row in user_profiles_df.iterrows()
}

recommended_item_ids = retrieval_results["item_id"].astype(str)
valid_recommended_items = set(games_df["item_id"].astype(str))

already_consumed_count = int(
    retrieval_results.apply(
        lambda row: str(row["item_id"]) in user_to_consumed_ids.get(row["user_id"], set()),
        axis=1,
    ).sum()
)
missing_item_ids_count = int((~recommended_item_ids.isin(valid_recommended_items)).sum())

assert not retrieval_results.empty, "retrieval_results está vacío."
assert already_consumed_count == 0, "Hay recomendaciones con juegos ya consumidos por el usuario."
assert missing_item_ids_count == 0, "Hay item_id recomendados que no existen en games_df."

print("Comprobaciones superadas correctamente.")
print("Recomendaciones ya consumidas:", already_consumed_count)
print("Item_id fuera de games_df:", missing_item_ids_count)

Comprobaciones superadas correctamente.
Recomendaciones ya consumidas: 0
Item_id fuera de games_df: 0


## 16. Conclusión

En esta sesión se han construido perfiles de usuario a partir de los juegos consumidos. Estos perfiles se han usado como consultas semánticas para recuperar candidatos similares en el espacio vectorial. FAISS ha devuelto un ranking top-k de juegos candidatos para cada usuario y estos candidatos serán la entrada del LLM en la sesión 3 para hacer re-ranking, justificación y filtro de alucinaciones.